# Volatility surfaces (strike × expiry grids)

`VolSurface` is a strike-by-expiry quote grid. `FxDeltaVolSurface` stores ATM / risk-reversal / butterfly quotes; evaluate those quotes with `get_fx_delta_vol` (or materialize a strike grid with `materialize_fx_delta_surface`) from `finstack_quant.models.volatility`.

## Concept

A **vol surface** attaches a Black–Scholes / Black **implied volatility** to each **(expiry, strike)** pair on a grid. **Smile** is how vol changes with strike at fixed expiry; **term structure** is how ATM vol changes with expiry. Surfaces are usually stored as **regular grids** with vols in **row-major** order for fast lookup and scenario bumps.

## API walkthrough

`VolSurface` is a data-only artifact in `finstack_quant.core.market_data`. Build it from expiry and strike axes plus a flat row-major volatility grid, then evaluate it with `get_surface_vol` or `get_surface_vol_clamped` from `finstack_quant.models.volatility`.

In [ ]:
from finstack_quant.core.market_data import VolSurface
from finstack_quant.models.volatility import get_surface_vol, get_surface_vol_clamped

surface = VolSurface(
    "SPX-IMPVOL",
    [0.25, 0.5, 1.0],
    [90.0, 100.0, 110.0],
    [0.24, 0.21, 0.23, 0.25, 0.22, 0.24, 0.26, 0.23, 0.25],
    secondary_axis="strike",
    interpolation_mode="vol",
    quote_type="black_lognormal",
)
print("surface:", surface)
print("grid_shape:", surface.grid_shape)
print("secondary_axis:", surface.secondary_axis, "interpolation_mode:", surface.interpolation_mode)
print("quote_type:", surface.quote_type)
print("ATM 6M vol:", round(get_surface_vol(surface, 0.5, 100.0), 4))
print("Interpolated 9M, K=105 vol:", round(get_surface_vol(surface, 0.75, 105.0), 4))

## Practical example

**Synthetic grid (pure Python):** build a tiny **strike × expiry** vol matrix and **lookup** by nearest indices — mirrors how bindings expose row-major flat arrays.

In [ ]:
from finstack_quant.core.market_data import MarketContext

ctx = MarketContext()
ctx.insert(surface)
stored = ctx.get_surface("SPX-IMPVOL")
print("Stored surface grid:", stored.grid_shape)
print("1Y smile slice:")
for strike in (90.0, 100.0, 110.0):
    print(f"  K={strike:>5.1f}  vol={get_surface_vol(stored, 1.0, strike):.4f}")
print("Clamped 18M, K=120 vol:", round(get_surface_vol_clamped(stored, 1.5, 120.0), 4))

## Model-free static arbitrage checks

Use `finstack_quant.models.volatility` helpers on plain grids (no `VolSurface` required) to test for butterfly, calendar-spread, and Dupire local-vol density issues before feeding a surface into pricing or risk.


In [ ]:
from finstack_quant.models.volatility import check_surface_grid

# Tiny 2-expiry x 3-strike grid (vols in decimal). Forwards broadcast or per-expiry.
strikes = [90.0, 100.0, 110.0]
expiries = [0.5, 1.0]
vols = [
    [0.24, 0.21, 0.23],  # 6M
    [0.25, 0.22, 0.24],  # 1Y
]
forwards = [100.0, 100.0]

report = check_surface_grid(strikes, expiries, vols, forward_prices=forwards)
print("passed:", report.passed)
print("total_violations:", report.total_violations)
print("by_type:", report.by_type)
print("violations (first 2):", report.violations[:2])

# Individuals return lists of violation dicts directly:
from finstack_quant.models.volatility import check_butterfly_grid
print("butterfly list len:", len(check_butterfly_grid(strikes, expiries, vols, forwards)))

## Takeaways

- `VolSurface` stores a full expiry × strike grid and exposes both checked lookup and clamped edge evaluation.
- `secondary_axis` keeps the surface semantics explicit, which matters when the second dimension is tenor instead of strike.
- `MarketContext` can now carry these surfaces directly, so option-style examples can share the same market snapshot model as curves and FX.

## Analyst program: total variance and FX delta quotes

Calendar-arbitrage checks compare total variance at fixed moneyness under the supplied forwards. A decreasing volatility by itself is not a violation. FX wing deltas use the documented forward-delta conversion; premium-adjusted conventions require a different conversion.

In [ ]:
import math
from finstack_quant.core.market_data import VolSurface, FxDeltaVolSurface
from finstack_quant.models.volatility import get_surface_vol, check_surface_grid, delta_to_strike, get_fx_delta_vol

surface = VolSurface('LESSON-VOL', [0.5, 1.0], [90.0, 100.0, 110.0],
    [[0.20, 0.20, 0.20], [0.25, 0.25, 0.25]], interpolation_mode='total_variance')
interpolated = get_surface_vol(surface, 0.75, 100.0)
assert abs(interpolated ** 2 * 0.75 - (0.20 ** 2 * 0.5 + 0.25 ** 2 * 1.0) / 2) < 1e-12
broken = check_surface_grid([90.0, 100.0, 110.0], [0.5, 1.0],
    [[0.40] * 3, [0.10] * 3], forward_prices=[100.0, 100.0])
assert not broken.passed and broken.total_violations > 0
fx_surface = FxDeltaVolSurface('EURUSD-VOL', [0.5, 1.0], [0.10, 0.11], [0.01, 0.012], [0.003, 0.004])
call_vol = 0.11 + 0.004 + 0.012 / 2
strike = delta_to_strike(0.25, 1.08, call_vol, 1.0)
resolved = get_fx_delta_vol(fx_surface, 1.0, strike, 1.08)
assert strike > 1.08 and abs(resolved - call_vol) < 1e-8
print({'interpolated_vol': interpolated, 'fx_25d_call_strike': strike, 'violations': broken.by_type})

## Declining total variance

In [ ]:
from finstack_quant.models.volatility import check_surface_grid
short_t,long_t=0.5,1.0
short_vol,long_vol=0.4,0.1
report=check_surface_grid([90.,100.,110.],[short_t,long_t],[[short_vol]*3,[long_vol]*3],forward_prices=[100.,100.])
assert short_vol**2*short_t > long_vol**2*long_t
assert not report.passed and report.by_type['calendar_spread']>0
print({'short_total_variance':short_vol**2*short_t,'long_total_variance':long_vol**2*long_t,'violations':report.by_type})
